In [ ]:
# General libraries
import os
import re
import random
import pickle
import statistics
import collections
from collections import Counter
from itertools import combinations

# Data handling
import numpy as np
import pandas as pd

# Visualization
import seaborn as sns
from matplotlib import pyplot as plt, cm, colors, colorbar
from matplotlib_venn import venn2
from mpl_toolkits.axes_grid1 import make_axes_locatable
from adjustText import adjust_text

# Machine learning & preprocessing
from sklearn.linear_model import LinearRegression, RidgeClassifier, LogisticRegression, SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.manifold import TSNE
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, silhouette_score,
    davies_bouldin_score, calinski_harabasz_score, pairwise_distances
)

# Dimensionality reduction
import umap
import umap.umap_ as umap_module  # if you need the lower-level API

# Feature selection
from boruta import BorutaPy

# Shapelet learning
from pyts.classification import LearningShapelets
from pyts.datasets import load_gunpoint
from pyts.utils import windowed_view

# Statistical tests and models
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from statannot import add_stat_annotation
#import scikit_posthocs as sp
import shap
#from xgboost import XGBRegressor
from scipy import stats
from scipy.stats import (
    ttest_ind, levene, mannwhitneyu, shapiro, mstats,
    pearsonr, kruskal, skew
)
from scipy.spatial import ConvexHull, convex_hull_plot_2d

# Progress bar
from tqdm import tqdm
from statsmodels.formula.api import ols
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore') 
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal, ttest_ind, f_oneway
from statsmodels.stats.multitest import multipletests
import warnings
from scipy.stats import chi2_contingency
from collections import Counter

In [ ]:
org_directory="/home/jupy/Data_SourceFC/T0"
ls_org_directory=os.listdir(org_directory)
directory_T0=[item for item in ls_org_directory if item !='.ipynb_checkpoints' ]
directory_T0_ob=[x for x in directory_T0 if x.startswith("F")]
directory_T0_lean=[x for x in directory_T0 if x.startswith("L")]

org_directory="/home/jupy/Data_SourceFC/T45"
ls_org_directory=os.listdir(org_directory)
directory_T45=[item for item in ls_org_directory if item != '.ipynb_checkpoints']
directory_T45_ob=[x for x in directory_T45 if x.startswith("F")]
directory_T45_lean=[x for x in directory_T45 if x.startswith("L")]

def extract_key(filename):
    return filename[0:4]
    
def align_subjects_btw_T0_T45 (directory_list_T0, directory_list_T45):
    # Create dictionaries for all time points
    time_points = {
        'T0': {extract_key(f): f for f in directory_list_T0 if extract_key(f)},
        'T45': {extract_key(f): f for f in directory_list_T45 if extract_key(f)}}    
    # Find common keys across ALL time points
    common_keys = set(time_points['T0'])  # Start with T0 keys
    for tp in time_points:
        common_keys &= set(time_points[tp])  # Intersect with each time point
        # Extract aligned files for each time point (sorted by key)
    aligned_files = {
        tp: [time_points[tp][key] for key in sorted(common_keys)]
        for tp in time_points }
    # Find unaligned files for each time point
    unique_files = {
        tp: [f for key, f in time_points[tp].items() if key not in common_keys]
        for tp in time_points}
    
    excl0=unique_files['T0']
    aligned_directory_T0=[item for item in directory_list_T0 if item not in excl0 ]
    excl45=unique_files['T45']
    aligned_directory_T45=[item for item in directory_list_T45 if item not in excl45]
    print ('Length T0: ',len(aligned_directory_T0),'    Length T45',len(aligned_directory_T45)) 
    
    # Create dictionaries with 4-digit keys
    dict_T0 = {extract_key(f): f for f in aligned_directory_T0 if extract_key(f)}
    dict_T45 = {extract_key(f): f for f in aligned_directory_T45 if extract_key(f)}
    # Find aligned keys
    common_keys = set(dict_T0) & set(dict_T45)
    aligned_T0 = [dict_T0[key] for key in common_keys]
    aligned_T45 = [dict_T45[key] for key in common_keys]
    # Find unaligned elements
    unique_T0 = [f for key, f in dict_T0.items() if key not in common_keys]
    unique_T45 = [f for key, f in dict_T45.items() if key not in common_keys]
    # Sort both aligned lists by key (first 4 digits)
    aligned_pairs = sorted(zip(aligned_T0, aligned_T45), key=lambda x: extract_key(x[0]))
    aligned_T0_sorted, aligned_T45_sorted = zip(*aligned_pairs) if aligned_pairs else ([], [])
    # Final sorted lists (now aligned by first 4 digits)
    finaldir_T0 = list(aligned_T0_sorted) 
    finaldir_T45 = list(aligned_T45_sorted) 
    
    return finaldir_T0, finaldir_T45

In [ ]:
finaldir_T0_lean,finaldir_T45_lean=align_subjects_btw_T0_T45 (directory_T0_lean, directory_T45_lean)
finaldir_T0_ob,finaldir_T45_ob=align_subjects_btw_T0_T45 (directory_T0_ob, directory_T45_ob)

In [ ]:
def process_demo_data (demo_df):
    sorted_demo_data = demo_df.replace('#NULL!', np.nan)
    for col in sorted_demo_data.columns:
        sorted_demo_data[col] = pd.to_numeric(sorted_demo_data[col], errors='ignore')
    sorted_demo_data = sorted_demo_data.fillna(sorted_demo_data.mean(numeric_only=True))
    return sorted_demo_data
    
demo_data=pd.read_csv('/home/jupy/Subtypes_Obesity_Clustering/DemographicalData_BL.csv')
demo_data=demo_data.drop(columns=['ADD_SLAIENCE2','ADD_EUPHORIA2','ADD_TOLERANCE2','ADD_WITHDRAWAL2',
 'ADD_CONFLIC12','ADD_CONFLICT22', 'ADD_RELAPSE2','ADD_IDENTIFICATION2'])

demodf_lean=demo_data[demo_data['scr_no'].isin( [f[0:3] for f in finaldir_T0_lean] )] # f[0:3] becuase lean people ID e.g. L03 is 3 digit
demodf_ob=demo_data[demo_data['scr_no'].isin( [f[0:4] for f in finaldir_T0_ob] )] # f[0:4] becuase obese people ID e.g. F246 is 4 digit
#print([f[0:4] for f in finaldir_T0_ob]==[f[0:4] for f in finaldir_T45_ob]) # if True means T0 and T45 have the same ID of subject  

sorted_demo_data_ob=process_demo_data (demodf_ob).reset_index(drop=True)
sorted_demo_data_lean=process_demo_data (demodf_lean).reset_index(drop=True)

noal_sorted_demo_data_lean=sorted_demo_data_lean
noal_sorted_demo_data_ob=sorted_demo_data_ob

scr_list = noal_sorted_demo_data_lean['scr_no'].tolist()
noal_finaldir_T0_lean=[f for f in finaldir_T0_lean if any(scr in f for scr in scr_list)] 
noal_finaldir_T45_lean=[f for f in finaldir_T45_lean if any(scr in f for scr in scr_list)] 

scr_list = noal_sorted_demo_data_ob['scr_no'].tolist()
noal_finaldir_T0_ob = [f for f in finaldir_T0_ob if any(scr in f for scr in scr_list)] 
noal_finaldir_T45_ob = [f for f in finaldir_T45_ob if any(scr in f for scr in scr_list)] 

from scipy.stats import zscore
noal_sorted_demo_data_ob=noal_sorted_demo_data_ob.drop(columns=['Category','Reproductive_status']) #'scr_no',

for i in [ 'BL_SATIS15', 'ADD_CHOC' , 'ADD_EUPHORIA' ]:
    noal_sorted_demo_data_ob[i]=noal_sorted_demo_data_ob[i].round().astype(int)
noal_sorted_demo_data_ob=noal_sorted_demo_data_ob.drop(columns=['IE_TOTAL'])

In [ ]:
cluster_label_satiety_delta=[
    0, 0, 0, 1, 1, 1, 1, 1, 1, 1,
    0, 0, 0, 0, 0, 1, 1, 0, 1, 0,
    1, 1, 0, 1, 0, 1, 1, 0, 0, 1]

cluster_label_satiety_theta=[2, 1, 1, 1, 0, 2, 2, 0, 0, 1, 1, 2, 1, 0, 1, 2, 1, 2, 0, 1, 0, 2, 1, 1, 0, 1, 2, 0, 2, 2]

cluster_label_satiety_alpha=[ 0, 1, 0, 0, 1,
    1, 1, 1, 0, 1,
    1, 1, 0, 0, 1,
    0, 1, 0, 0, 1,
    1, 0, 1, 0, 0,
    0, 1, 1, 0, 1]

cluster_label_satiety_beta= [  1, 1, 0, 1, 0, 0, 1, 0, 1, 1,
    0, 0, 0, 1, 0, 1, 1, 0, 0, 1,
    1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

cluster_label_satiety_gamma= [0, 0, 0, 0, 1, 1, 1, 1, 0, 1,
    1, 0, 1, 1, 1, 0, 1, 1, 1, 0,
    1, 1, 1, 0, 1, 1, 0, 1, 0, 0 ]

cluster_label_satiety_cfc = [
    0, 1, 0, 1, 1, 0, 0, 1, 0, 1,
    0, 0, 1, 0, 1, 0, 1, 0, 0, 1,
    0, 1, 1, 1, 0, 1, 0, 0, 0, 1
]

In [ ]:
cluster_label_fasting_delta = [1,0,1,1,0,0,0,0,0,1,1,2,1,0,2,1,2,0,2,1,2,0,1,1,1,1,0,0,2,0]

cluster_label_fasting_theta = [0,0,0,1,1,0,1,1,1,1,0,0,0,1,1,1,0,0,0,0,0,0,0,0,1,1,0,1,0,0]

cluster_label_fasting_alpha = [1,0,0,1,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,1,1,1,0,1,1,0,0,0,0,1]

cluster_label_fasting_beta = [1,0,1,1,0,1,1,1,1,1,1,0,1,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,1]

cluster_label_fasting_gamma = [0,1,0,0,0,0,1,1,0,1,1,0,0,0,1,0,0,1,0,1,1,1,1,1,0,0,0,1,0,1]

cluster_label_fasting_cfc= [
    1, 1, 1, 0, 0, 1, 0, 0, 1, 0,
    0, 1, 1, 1, 0, 0, 1, 0, 1, 1,
    0, 1, 0, 0, 0, 1, 1, 0, 1, 0
]

In [ ]:
cluster_dict = {
    "cluster_label_satiety_delta": cluster_label_satiety_delta,
    "cluster_label_satiety_theta": cluster_label_satiety_theta,
    "cluster_label_satiety_alpha": cluster_label_satiety_alpha,
    "cluster_label_satiety_beta": cluster_label_satiety_beta,
    "cluster_label_satiety_gamma": cluster_label_satiety_gamma,
    "cluster_label_satiety_cfc": cluster_label_satiety_cfc,
    
    "cluster_label_fasting_delta": cluster_label_fasting_delta,
    "cluster_label_fasting_theta": cluster_label_fasting_theta,
    "cluster_label_fasting_alpha": cluster_label_fasting_alpha,
    "cluster_label_fasting_beta": cluster_label_fasting_beta,
    "cluster_label_fasting_gamma": cluster_label_fasting_gamma,
    "cluster_label_fasting_cfc": cluster_label_fasting_cfc
}

# Add each list as a column
for col_name, values in cluster_dict.items():
    noal_sorted_demo_data_ob[col_name] = values

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations

def cohens_d(x, y):
    """Cohen's d for two independent groups."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan
    
    sx, sy = np.std(x, ddof=1), np.std(y, ddof=1)
    pooled_sd = np.sqrt(((nx - 1) * sx**2 + (ny - 1) * sy**2) / (nx + ny - 2))
    
    if pooled_sd == 0:
        return np.nan
    
    return (np.mean(x) - np.mean(y)) / pooled_sd


def cliffs_delta(x, y):
    """Cliff's delta for two independent groups."""
    x = np.asarray(x)
    y = np.asarray(y)
    
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return np.nan
    
    gt = 0
    lt = 0
    for xi in x:
        gt += np.sum(xi > y)
        lt += np.sum(xi < y)
    
    return (gt - lt) / (nx * ny)


def cramers_v(confusion_matrix):
    """Cramér's V for association between two categorical variables."""
    chi2 = stats.chi2_contingency(confusion_matrix, correction=False)[0]
    n = confusion_matrix.to_numpy().sum()
    if n == 0:
        return np.nan
    
    r, k = confusion_matrix.shape
    denom = min(r - 1, k - 1)
    if denom == 0:
        return np.nan
    
    return np.sqrt(chi2 / (n * denom))


def eta_squared_from_anova(groups):
    """
    Eta-squared from one-way ANOVA:
    eta² = SS_between / SS_total
    """
    clean_groups = [np.asarray(g, dtype=float) for g in groups if len(g) > 0]
    if len(clean_groups) < 2:
        return np.nan
    
    all_vals = np.concatenate(clean_groups)
    grand_mean = np.mean(all_vals)
    
    ss_between = sum(len(g) * (np.mean(g) - grand_mean) ** 2 for g in clean_groups)
    ss_total = sum(np.sum((g - grand_mean) ** 2) for g in clean_groups)
    
    if ss_total == 0:
        return np.nan
    
    return ss_between / ss_total


def epsilon_squared_from_kruskal(groups):
    """
    Epsilon-squared for Kruskal-Wallis:
    epsilon² = (H - k + 1) / (n - k)
    """
    clean_groups = [np.asarray(g) for g in groups if len(g) > 0]
    k = len(clean_groups)
    n = sum(len(g) for g in clean_groups)
    
    if k < 2 or n <= k:
        return np.nan
    
    h_stat, _ = stats.kruskal(*clean_groups)
    return (h_stat - k + 1) / (n - k)


def get_cluster_col(state, band):
    """
    Build cluster column name automatically from state and band.
    Example:
        state='fasting', band='delta'
        -> 'cluster_label_fasting_delta'
    """
    state = state.strip().lower()
    band = band.strip().lower()
    return f"cluster_label_{state}_{band}"


def compute_cluster_effect_size(
    df,
    variable,
    variable_type,
    measure=None,
    state=None,
    band=None,
    cluster_col=None,
    dropna=True,
    return_pairwise_for_two_cluster_only=True
):
    """
    Compute effect size of a variable across cluster groups.

    Parameters
    ----------
    df : pandas.DataFrame
    variable : str
        Column name of the variable to test.
    variable_type : str
        One of: 'numerical', 'ordinal', 'categorical'
    measure : str or None
        Effect size measure to use.
        Allowed:
            - numerical, 2 clusters: 'cohens_d'
            - ordinal, 2 clusters: 'cliffs_delta'
            - >2 clusters parametric: 'eta_squared'
            - >2 clusters nonparametric: 'epsilon_squared'
            - categorical: 'cramers_v'
        If None, defaults are chosen from variable_type and number of clusters.
    state : str or None
        'fasting' or 'satiety'
    band : str or None
        e.g. 'delta', 'theta', 'alpha', 'beta', 'gamma', 'cfc'
    cluster_col : str or None
        If given, overrides state/band and uses this cluster column directly.
    dropna : bool
        Whether to drop missing rows in variable/cluster column.
    return_pairwise_for_two_cluster_only : bool
        For 2-cluster data, returns the group labels used.

    Returns
    -------
    dict
        Summary of effect size result.
    """
    variable_type = variable_type.strip().lower()
    
    if cluster_col is None:
        if state is None or band is None:
            raise ValueError("Provide either cluster_col OR both state and band.")
        cluster_col = get_cluster_col(state, band)
    
    if cluster_col not in df.columns:
        raise ValueError(f"Cluster column '{cluster_col}' not found in DataFrame.")
    
    if variable not in df.columns:
        raise ValueError(f"Variable '{variable}' not found in DataFrame.")
    
    tmp = df[[variable, cluster_col]].copy()
    
    if dropna:
        tmp = tmp.dropna(subset=[variable, cluster_col])
    
    cluster_values = sorted(tmp[cluster_col].unique())
    n_clusters = len(cluster_values)
    
    if n_clusters < 2:
        raise ValueError("Need at least 2 clusters to compute an effect size.")
    
    # Default measure selection
    if measure is None:
        if variable_type == "categorical":
            measure = "cramers_v"
        elif n_clusters == 2:
            if variable_type == "numerical":
                measure = "cohens_d"
            elif variable_type == "ordinal":
                measure = "cliffs_delta"
            else:
                raise ValueError(
                    "For 2 clusters, variable_type must be 'numerical', 'ordinal', or use 'categorical' with cramers_v."
                )
        else:
            if variable_type == "numerical":
                measure = "eta_squared"
            elif variable_type == "ordinal":
                measure = "epsilon_squared"
            else:
                raise ValueError(
                    "For >2 clusters, variable_type must be 'numerical', 'ordinal', or use 'categorical' with cramers_v."
                )
    
    measure = measure.strip().lower()
    
    result = {
        "variable": variable,
        "variable_type": variable_type,
        "cluster_column": cluster_col,
        "n_clusters": n_clusters,
        "clusters": cluster_values,
        "measure": measure,
        "effect_size": None
    }
    
    # Categorical variable -> Cramér's V
    if measure == "cramers_v":
        contingency = pd.crosstab(tmp[cluster_col], tmp[variable])
        effect = cramers_v(contingency)
        result["effect_size"] = effect
        result["contingency_table"] = contingency
        return result
    
    # Prepare grouped data
    groups = [tmp.loc[tmp[cluster_col] == c, variable].values for c in cluster_values]
    
    # Two-cluster measures
    if measure == "cohens_d":
        if n_clusters != 2:
            raise ValueError("Cohen's d is for 2-cluster comparisons only.")
        effect = cohens_d(groups[0], groups[1])
        result["effect_size"] = effect
        if return_pairwise_for_two_cluster_only:
            result["group_comparison"] = (cluster_values[0], cluster_values[1])
        return result
    
    if measure == "cliffs_delta":
        if n_clusters != 2:
            raise ValueError("Cliff's delta is for 2-cluster comparisons only.")
        effect = cliffs_delta(groups[0], groups[1])
        result["effect_size"] = effect
        if return_pairwise_for_two_cluster_only:
            result["group_comparison"] = (cluster_values[0], cluster_values[1])
        return result
    
    # Multi-cluster measures
    if measure == "eta_squared":
        if n_clusters < 3:
            raise ValueError("eta_squared is intended for comparisons involving more than 2 clusters.")
        effect = eta_squared_from_anova(groups)
        result["effect_size"] = effect
        return result
    
    if measure == "epsilon_squared":
        if n_clusters < 3:
            raise ValueError("epsilon_squared is intended for comparisons involving more than 2 clusters.")
        effect = epsilon_squared_from_kruskal(groups)
        result["effect_size"] = effect
        return result
    
    raise ValueError(
        "Invalid measure. Choose from: "
        "'cohens_d', 'cliffs_delta', 'eta_squared', 'epsilon_squared', 'cramers_v'."
    )

In [ ]:
cols_to_log = [
    'GhrelinB_0', 'GhrelinB_45',
    'glucB_0', 'glucB_45',
    'insB_0', 'insB_45',
    'GLP1B_0', 'GLP1B_45',
    'PYYB_0', 'PYYB_45'
]

for col in cols_to_log:
    noal_sorted_demo_data_ob[f'log_{col}'] = np.log(noal_sorted_demo_data_ob[col])

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt

def plot_ancova_by_cluster(df,
                          value_col,
                          cluster_col,
                          covariates,
                          figsize=(4, 3)):

    cols = [value_col, cluster_col] + covariates
    plot_df = df[cols].dropna().copy()
    plot_df[cluster_col] = plot_df[cluster_col].astype("category")

    covariate_str = " + ".join(covariates)
    formula = f"{value_col} ~ C({cluster_col}) + {covariate_str}"

    model = smf.ols(formula, data=plot_df).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)

    p_val = anova_table.loc[f"C({cluster_col})", "PR(>F)"]

    if p_val < 0.001:
        sig_label = "***"
    elif p_val < 0.01:
        sig_label = "**"
    elif p_val < 0.05:
        sig_label = "*"
    else:
        sig_label = "ns"

    plt.figure(figsize=figsize)
    ax = sns.boxplot(data=plot_df, x=cluster_col, y=value_col)

    ax.set_xlabel("Cluster ID")
    ax.set_ylabel(value_col)

    y_max = plot_df[value_col].max()
    y_min = plot_df[value_col].min()
    y_range = y_max - y_min if y_max > y_min else 1

    line_y = y_max + y_range * 0.12
    text_y = y_max + y_range * 0.15
    x1, x2 = 0, len(plot_df[cluster_col].cat.categories) - 1

    ax.plot([x1, x1, x2, x2],
            [line_y - y_range * 0.01, line_y, line_y, line_y - y_range * 0.01],
            lw=1.8, c="black")

    ax.text((x1 + x2) / 2, text_y, sig_label,
            ha="center", va="bottom", fontsize=14)

    ax.set_ylim(y_min, y_max + y_range * 0.22)

    plt.tight_layout()
    plt.show()

    print(f"\nANCOVA result for {value_col}")
    print(f"Formula: {formula}")
    print(anova_table)

    return model, anova_table

In [ ]:
def plot_hormone_ancova_log(df,
                            hormone,
                            timepoint,
                            cluster_col,
                            figsize=(7, 5)):
    """
    Automatically uses log-transformed variables:
    log_Hormone_0 / log_Hormone_45
    """

    value_col = f"log_{hormone}_{timepoint}"

    if timepoint == 0:
        covariates = ["bmi", "age"]
    elif timepoint == 45:
        covariates = [f"log_{hormone}_0", "bmi", "age"]
    else:
        raise ValueError("timepoint must be 0 or 45")

    return plot_ancova_by_cluster(
        df=df,
        value_col=value_col,
        cluster_col=cluster_col,
        covariates=covariates,
        figsize=figsize
    )

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

def compute_ancova_effect_size(df,
                               value_col,
                               cluster_col,
                               covariates):
    """
    Returns partial eta squared for cluster effect
    """

    cols = [value_col, cluster_col] + covariates
    data = df[cols].dropna().copy()
    data[cluster_col] = data[cluster_col].astype("category")

    covariate_str = " + ".join(covariates)
    formula = f"{value_col} ~ C({cluster_col}) + {covariate_str}"

    model = smf.ols(formula, data=data).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)

    ss_effect = anova_table.loc[f"C({cluster_col})", "sum_sq"]
    ss_resid = anova_table.loc["Residual", "sum_sq"]

    eta_p = ss_effect / (ss_effect + ss_resid)

    return eta_p, anova_table

def compute_hormone_effect_size_log(df,
                                    hormone,
                                    timepoint,
                                    cluster_col):
    """
    Automatically applies correct model for 0 vs 45
    """

    value_col = f"log_{hormone}_{timepoint}"

    if timepoint == 0:
        covariates = ["bmi", "age"]
    elif timepoint == 45:
        covariates = [f"log_{hormone}_0", "bmi", "age"]
    else:
        raise ValueError("timepoint must be 0 or 45")

    eta_p, anova_table = compute_ancova_effect_size(
        df=df,
        value_col=value_col,
        cluster_col=cluster_col,
        covariates=covariates
    )

    return eta_p, anova_table

In [ ]:
hormones = ["GhrelinB", "glucB", "insB", "GLP1B", "PYYB"]

bandtosee='gamma'
for horm in hormones:
    print(f"\n{'='*70}\n{horm}_0\n{'='*70}")
    plot_hormone_ancova_log(
        noal_sorted_demo_data_ob,
        hormone=horm,
        timepoint=0,
        cluster_col=f"cluster_label_fasting_{bandtosee}"
    )

    print(f"\n{'='*70}\n{horm}_45\n{'='*70}")
    plot_hormone_ancova_log(
        noal_sorted_demo_data_ob,
        hormone=horm,
        timepoint=45,
        cluster_col=f"cluster_label_satiety_{bandtosee}"
    )

In [ ]:
hormones = ["GhrelinB", "glucB", "insB", "GLP1B", "PYYB"]
bandtosee = 'theta'

for horm in hormones:

    print(f"\n{'='*70}\n{horm}_0\n{'='*70}")
    eta_p, anova_table = compute_hormone_effect_size_log(
        noal_sorted_demo_data_ob,
        hormone=horm,
        timepoint=0,
        cluster_col=f"cluster_label_fasting_{bandtosee}"
    )
    print(f"Partial η²: {eta_p:.3f}")

    print(f"\n{'='*70}\n{horm}_45\n{'='*70}")
    eta_p, anova_table = compute_hormone_effect_size_log(
        noal_sorted_demo_data_ob,
        hormone=horm,
        timepoint=45,
        cluster_col=f"cluster_label_satiety_{bandtosee}"
    )
    print(f"Partial η²: {eta_p:.3f}")